In [ ]:
import geopandas as gpd

path = "/kaggle/input/datasets/antsasarobidyran/sfi-data/Data_SFI.gpkg"

import fiona
layers = fiona.listlayers(path)
print("Layers found:", layers)

gdf = gpd.read_file(path, layer=layers[0])

# Basic info
print("\nCRS:", gdf.crs)
print("Shape (rows, cols):", gdf.shape)
print("\nColumn names:")
print(gdf.columns.tolist())

print("\nData types:")
print(gdf.dtypes)

attribute_table = gdf.drop(columns="geometry")
print("\nAttribute table preview:")
display(attribute_table.head(20))

Layers found: ['sfi']

CRS: EPSG:4326
Shape (rows, cols): (307, 18)

Column names:
['Site', 'Plot', 'NAME', 'X', 'Y', 'Landuse', 'Reclass_Fire_frequency', 'ISF_a', 'ISF_b', 'ISF_c', 'ISF_d', 'ISF_e', 'CLASSE_ISF_a', 'CLASSE_ISF_b', 'CLASSE_ISF_c', 'CLASSE_ISF_d', 'CLASSE_ISF_e', 'geometry']

Data types:
Site                        object
Plot                         int32
NAME                        object
X                          float64
Y                          float64
Landuse                     object
Reclass_Fire_frequency      object
ISF_a                      float64
ISF_b                      float64
ISF_c                      float64
ISF_d                      float64
ISF_e                      float64
CLASSE_ISF_a                object
CLASSE_ISF_b                object
CLASSE_ISF_c                object
CLASSE_ISF_d                object
CLASSE_ISF_e                object
geometry                  geometry
dtype: object

Attribute table preview:


,Site,Plot,NAME,X,Y,Landuse,Reclass_Fire_frequency,ISF_a,ISF_b,ISF_c,ISF_d,ISF_e,CLASSE_ISF_a,CLASSE_ISF_b,CLASSE_ISF_c,CLASSE_ISF_d,CLASSE_ISF_e
0,AF,1,AF_1,46.795105,-16.312945,Forest,F0,0.342429,0.342429,0.200000,0.264017,0.342429,Bas,Bas,Tres bas,Bas,Bas
1,AF,10,AF_10,46.922265,-16.149362,Forest,F1,0.560993,0.509669,0.509669,0.509669,0.509669,Moyen,Moyen,Moyen,Moyen,Moyen
2,AF,100,AF_100,46.794242,-16.293132,Reforestation,F1,0.354547,0.406446,0.342429,0.342429,0.342429,Bas,Bas,Bas,Bas,Bas
3,AF,101,AF_101,46.796618,-16.293087,Reforestation,F1,0.650948,0.523489,0.381635,0.381635,0.381635,Moyen,Moyen,Bas,Bas,Bas
4,AF,102,AF_102,46.793896,-16.291441,Reforestation,F1,0.342429,0.381635,0.381635,0.381635,0.381635,Bas,Bas,Bas,Bas,Bas
5,AF,103,AF_103,46.795386,-16.290423,Reforestation,F0,0.457770,0.381635,0.406446,0.381635,0.381635,Bas,Bas,Bas,Bas,Bas
6,AF,104,AF_104,46.795045,-16.288118,Reforestation,F0,NaN,0.381635,0.381635,0.381635,0.381635,None,Bas,Bas,Bas,Bas
7,AF,105,AF_105,46.793144,-16.289205,Reforestation,F0,0.381635,0.381635,0.445652,0.342429,0.342429,Bas,Bas,Bas,Bas,Bas
8,AF,106,AF_106,46.783392,-16.294873,Reforestation,F0,0.457770,0.420841,0.420841,0.420841,0.420841,Bas,Bas,Bas,Bas,Bas
9,AF,107,AF_107,46.779846,-16.300620,Reforestation,F0,0.342429,0.303223,0.303223,0.303223,0.303223,Bas,Bas,Bas,Bas,Bas


In [ ]:
import os, gc, time, csv, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.windows import Window
from joblib import Parallel, delayed
from tqdm import tqdm
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import cupy as cp
import cuml
from cuml.fil import ForestInference

ROI_PATH  = '/kaggle/input/datasets/antsasarobidyran/area-of-interest/AOI_dissolved.gpkg'
DATA_PATH = '/kaggle/input/datasets/antsasarobidyran/sfi-data/Data_SFI.gpkg'
CROPPED_DIR = '/kaggle/input/datasets/antsasarobidyran/tanety-covariates'
FINAL_GPKG = '/kaggle/working/Tanety/Data_covariates.gpkg'
RESULTS_DIR    = '/kaggle/working/Tanety/ML_Results'
PREDICTION_DIR = '/kaggle/working/Tanety/predictions_FIL'

COVARIATE_NAMES = ['chm', 'ELEV_ALOS', 'lulc', 'map_corrected', 'mat_corrected',
                    'NDVI', 'NDWI', 'NIRI', 'NPP', 'SLOPE_ALOS',
                    'soil_type', 'treecover', 'TWI']
CATEGORICAL_COVARIATES = ['lulc', 'soil_type']

# SFI depth layers: a=0-10cm, b=10-20cm, c=20-30cm, d=30-60cm, e=60-90cm
RESPONSES = ['ISF_a', 'ISF_b', 'ISF_c', 'ISF_d', 'ISF_e']

N_JOBS_CPU = -1
N_FOLDS_OUTER = 5
N_FOLDS_INNER = 3
OPTUNA_TRIALS_CV    = 50
OPTUNA_TRIALS_FINAL = 50
SEED = 123
TILE_SIZE       = 2048
NODATA_OUT      = -9999.0
RF_N_ESTIMATORS = None
FIL_BATCH_SIZE  = 600_000
GPU_ID          = 0

os.makedirs(os.path.dirname(FINAL_GPKG), exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(PREDICTION_DIR, exist_ok=True)
print("Config ready.")

Config ready.


In [ ]:
print("Loading ROI and data points ...")
roi  = gpd.read_file(ROI_PATH)
data = gpd.read_file(DATA_PATH)

print(roi)
print(data)
print(list(data.columns))

fig, ax = plt.subplots(figsize=(7, 7))
roi.plot(ax=ax, facecolor='grey', edgecolor='black', linewidth=0.5)
data.plot(ax=ax, color='red', markersize=15, alpha=0.7)
ax.set_title('Data points over ROI')
ax.set_axis_off()
plt.show()

In [7]:
# -----------------------------------------------------------------------
# LOAD SFI DATA POINTS
# -----------------------------------------------------------------------
data = gpd.read_file(DATA_PATH)
print(f"Loaded {len(data)} points from {DATA_PATH}")
print(data.head())

# -----------------------------------------------------------------------
# LIST COVARIATE RASTERS
# -----------------------------------------------------------------------
cropped_files = sorted(
    f for f in os.listdir(CROPPED_DIR) if f.lower().endswith(('.tif', '.tiff'))
)
cropped_paths = [os.path.join(CROPPED_DIR, f) for f in cropped_files]
print(f"Found {len(cropped_paths)} covariate rasters: {cropped_files}")

# -----------------------------------------------------------------------
# EXTRACT COVARIATE VALUES AT POINTS (parallel)
# -----------------------------------------------------------------------
def _extract_one_raster(cropped_path, data_gdf):
    name = os.path.splitext(os.path.basename(cropped_path))[0]
    with rasterio.open(cropped_path) as src:
        pts_proj = data_gdf.to_crs(src.crs)
        coords   = [(geom.x, geom.y) for geom in pts_proj.geometry]
        vals     = np.array([v[0] for v in src.sample(coords)], dtype=np.float64)
        nodata   = src.nodata
        if nodata is not None:
            vals[np.isclose(vals, nodata)] = np.nan
    return name, vals

print("\nExtracting covariate values at data points (parallel) ...")
extracted = Parallel(n_jobs=N_JOBS_CPU)(
    delayed(_extract_one_raster)(p, data) for p in cropped_paths
)

extracted_df = pd.DataFrame({name: vals for name, vals in extracted})
data_extracted = data.reset_index(drop=True).copy()
for col in extracted_df.columns:
    data_extracted[col] = extracted_df[col].values

print(data_extracted.head())

# -----------------------------------------------------------------------
# SUMMARY STATS FOR EXTRACTED COVARIATES
# -----------------------------------------------------------------------
covariate_names_extracted = list(extracted_df.columns)
summary_table = (
    extracted_df[covariate_names_extracted]
    .melt(var_name='variable', value_name='value')
    .groupby('variable')['value']
    .agg(min='min', max='max', mean='mean', median='median', std='std')
    .reset_index()
)
summary_table['cv'] = summary_table['std'] / summary_table['mean'] * 100
print("\nCovariate summary statistics:")
print(summary_table)

# -----------------------------------------------------------------------
# FINAL TABLE: SFI responses + covariates + geometry
# -----------------------------------------------------------------------
data_final = data_extracted.copy()
keep_cols = RESPONSES + COVARIATE_NAMES + ['geometry']
data_final = data_final[[c for c in keep_cols if c in data_final.columns]]
print(data_final.head())

data_final.to_file(FINAL_GPKG, driver='GPKG')
print(f"\nSaved final covariate table -> {FINAL_GPKG}")

Loaded 307 points from /kaggle/input/datasets/antsasarobidyran/sfi-data/Data_SFI.gpkg
  Site  Plot    NAME          X          Y        Landuse  \
0   AF     1    AF_1  46.795105 -16.312945         Forest   
1   AF    10   AF_10  46.922265 -16.149362         Forest   
2   AF   100  AF_100  46.794242 -16.293132  Reforestation   
3   AF   101  AF_101  46.796618 -16.293087  Reforestation   
4   AF   102  AF_102  46.793896 -16.291441  Reforestation   

  Reclass_Fire_frequency     ISF_a     ISF_b     ISF_c     ISF_d     ISF_e  \
0                     F0  0.342429  0.342429  0.200000  0.264017  0.342429   
1                     F1  0.560993  0.509669  0.509669  0.509669  0.509669   
2                     F1  0.354547  0.406446  0.342429  0.342429  0.342429   
3                     F1  0.650948  0.523489  0.381635  0.381635  0.381635   
4                     F1  0.342429  0.381635  0.381635  0.381635  0.381635   

  CLASSE_ISF_a CLASSE_ISF_b CLASSE_ISF_c CLASSE_ISF_d CLASSE_ISF_e  \
0       

# RF

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import optuna
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error


def calc_metrics(obs, pred):
    obs, pred = np.asarray(obs, dtype=np.float64), np.asarray(pred, dtype=np.float64)
    ok = np.isfinite(obs) & np.isfinite(pred)
    obs, pred = obs[ok], pred[ok]

    rmse = np.sqrt(mean_squared_error(obs, pred))
    mae  = mean_absolute_error(obs, pred)
    r2   = 1 - np.sum((obs - pred) ** 2) / np.sum((obs - obs.mean()) ** 2)
    iqr  = np.percentile(obs, 75) - np.percentile(obs, 25)
    rpiq = iqr / rmse if rmse > 0 else 0.0

    mx, my = obs.mean(), pred.mean()
    vx, vy = obs.var(ddof=0), pred.var(ddof=0)      
    sxy    = np.cov(obs, pred, ddof=0)[0, 1]
    ccc_den = vx + vy + (mx - my) ** 2
    ccc = (2 * sxy) / ccc_den if ccc_den > 0 else 0.0

    return dict(R2=r2, RMSE=rmse, MAE=mae, RPIQ=rpiq, CCC=ccc)


def impute_train_test(X_tr, X_te, covariate_names):
    """Median-impute covariates using TRAIN medians only, applied to both
    splits -- fixes the leakage from computing medians on the full dataset
    before the outer CV split. Falls back to 0 if a covariate is entirely
    NaN in the training fold."""
    tr_df = pd.DataFrame(X_tr, columns=covariate_names)
    te_df = pd.DataFrame(X_te, columns=covariate_names)
    medians = tr_df.median()
    still_na = medians[medians.isna()].index
    if len(still_na):
        medians[still_na] = 0.0
    return tr_df.fillna(medians).values, te_df.fillna(medians).values


def make_rf_objective(X_np, y_np, n_inner=N_FOLDS_INNER):
    def objective(trial):
        params = dict(
            n_estimators      = trial.suggest_int('n_estimators', 100, 600),
            max_depth         = trial.suggest_int('max_depth', 5, 30),
            min_samples_split = trial.suggest_int('min_samples_split', 2, 15),
            min_samples_leaf  = trial.suggest_int('min_samples_leaf', 1, 8),
            max_features      = trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
            random_state      = 42,
            n_jobs            = -1,
        )
        model    = RandomForestRegressor(**params)
        inner_kf = KFold(n_splits=n_inner, shuffle=True, random_state=0)
        scores   = []
        for tr, va in inner_kf.split(X_np):
            model.fit(X_np[tr], y_np[tr])
            scores.append(np.sqrt(mean_squared_error(y_np[va], model.predict(X_np[va]))))
        return float(np.mean(scores))
    return objective


def tune_rf(X_np, y_np, n_trials):
    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(make_rf_objective(X_np, y_np), n_trials=n_trials,
                    n_jobs=N_JOBS_CPU, show_progress_bar=False)
    return {**study.best_params, 'random_state': 42, 'n_jobs': -1}


def run_nested_cv(df, response, covariates, outer_k=N_FOLDS_OUTER,
                   n_trials=OPTUNA_TRIALS_CV, seed=SEED):
    sub = df.dropna(subset=[response]).reset_index(drop=True)

    kf = KFold(n_splits=outer_k, shuffle=True, random_state=seed)
    outer_metrics, fold_predictions, best_params_list = [], [], []

    X_all_raw = sub[covariates].values
    y_all = sub[response].values.astype(np.float64)

    for i, (tr_idx, te_idx) in enumerate(kf.split(X_all_raw), 1):
        # covariate imputation fit on TRAIN ONLY -- no leakage into test
        X_tr, X_te = impute_train_test(X_all_raw[tr_idx], X_all_raw[te_idx], covariates)
        y_tr, y_te = y_all[tr_idx], y_all[te_idx]

        best_params = tune_rf(X_tr, y_tr, n_trials)
        rf = RandomForestRegressor(**best_params)
        rf.fit(X_tr, y_tr)
        preds = rf.predict(X_te)

        m = calc_metrics(y_te, preds)
        m['fold'] = i
        outer_metrics.append(m)
        fold_predictions.append(pd.DataFrame({'fold': i, 'obs': y_te, 'pred': preds}))
        best_params_list.append({**best_params, 'fold': i})

    return dict(
        metrics=pd.DataFrame(outer_metrics),
        predictions=pd.concat(fold_predictions, ignore_index=True),
        best_params=pd.DataFrame(best_params_list),
    )


t0 = time.time()
cv_results = {r: run_nested_cv(data_final.drop(columns='geometry'), r, COVARIATE_NAMES) for r in RESPONSES}
print(f"Nested CV finished in {(time.time()-t0)/60:.2f} min")

summary_table_cv = pd.DataFrame([
    {'response': r, **{f'{k}_mean': cv_results[r]['metrics'][k].mean() for k in ['R2','RMSE','MAE','RPIQ','CCC']},
     **{f'{k}_sd': cv_results[r]['metrics'][k].std() for k in ['R2','RMSE','MAE','RPIQ','CCC']}}
    for r in RESPONSES
])
print(summary_table_cv)
summary_table_cv.to_csv(os.path.join(RESULTS_DIR, 'cv_summary.csv'), index=False)

obs_pred_all = pd.concat(
    [cv_results[r]['predictions'].assign(response=r) for r in RESPONSES], ignore_index=True
)
fig, axes = plt.subplots(1, len(RESPONSES), figsize=(9, 4.5))
for ax, r in zip(axes, RESPONSES):
    sub = obs_pred_all[obs_pred_all['response'] == r]
    ax.scatter(sub['obs'], sub['pred'], alpha=0.6, color='steelblue')
    lims = [sub[['obs', 'pred']].min().min(), sub[['obs', 'pred']].max().max()]
    ax.plot(lims, lims, linestyle='--', color='red')
    ax.set_title(r); ax.set_xlabel('Observed'); ax.set_ylabel('Predicted')
fig.suptitle('Observed vs Predicted (out-of-fold, nested CV)')
plt.tight_layout()
plt.show()

best_params_table = pd.concat(
    [cv_results[r]['best_params'].assign(response=r) for r in RESPONSES], ignore_index=True
).sort_values(['response', 'fold'])
best_params_table.to_csv(os.path.join(RESULTS_DIR, 'best_params_per_fold.csv'), index=False)

In [ ]:
# =============================================================================
# FINAL RF MODELS ON FULL DATA + PERMUTATION IMPORTANCE 
# =============================================================================

import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_squared_error

ARIAL_PATH = '/kaggle/input/datasets/hammaadali/arial-font/arial.ttf'

if os.path.exists(ARIAL_PATH):
    fm.fontManager.addfont(ARIAL_PATH)
    font_prop   = fm.FontProperties(fname=ARIAL_PATH)
    font_family = font_prop.get_name()
    print(f"Arial loaded: {font_family}")
else:
    print(f"Arial not found at {ARIAL_PATH} — falling back to sans-serif")
    font_family = 'sans-serif'

plt.rcParams.update({
    'font.family':       font_family,
    'font.size':         19,
    'axes.labelsize':    20,
    'axes.titlesize':    21,
    'xtick.labelsize':   18,
    'ytick.labelsize':   18,
    'axes.linewidth':    1.8,
    'xtick.major.width': 1.8,
    'ytick.major.width': 1.8,
    'xtick.major.size':  6,
    'ytick.major.size':  6,
    'pdf.fonttype':      42,
    'ps.fonttype':       42,
})

# Depth-labeled titles for the SFI responses
RESP_DISPLAY = {
    'ISF_a': r'SFI$_{0-10}$',
    'ISF_b': r'SFI$_{10-20}$',
    'ISF_c': r'SFI$_{20-30}$',
    'ISF_d': r'SFI$_{30-60}$',
    'ISF_e': r'SFI$_{60-90}$',
}
PANEL_LABELS = ['a.', 'b.', 'c.', 'd.', 'e.']
TOP_N        = len(COVARIATE_NAMES)  
N_REPEATS    = 20                     

COVARIATE_DISPLAY = {
    'chm':            'CHM',
    'ELEV_ALOS':      'Elevation',
    'lulc':           'LULC',
    'map_corrected':  'MAP',
    'mat_corrected':  'MAT',
    'NDVI':           'NDVI',
    'NDWI':           'NDWI',
    'NIRI':           'NIRI',
    'NPP':            'NPP',
    'SLOPE_ALOS':     'Slope',
    'soil_type':      'Soil Type',
    'treecover':      'Tree Cover',
    'TWI':            'TWI',
}


def spine_cleanup(ax):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    for sp in ['left', 'bottom']:
        ax.spines[sp].set_color('black')
        ax.spines[sp].set_linewidth(1.8)
    ax.tick_params(axis='both', colors='black', direction='out', length=6, width=1.8)
    ax.grid(False)


def draw_lollipop(ax, feature_cols, mean_vals, std_vals, top_n, xlabel,
                   label_map=None):
    paired = sorted(zip(feature_cols, mean_vals, std_vals),
                     key=lambda x: x[1], reverse=True)[:top_n]
    keys   = [k for k, _, _ in paired]
    vals   = np.array([v for _, v, _ in paired])
    stds   = np.array([s for _, _, s in paired])
    n      = len(paired)
    ypos   = np.arange(n)[::-1]

    for y, val, std in zip(ypos, vals, stds):
        ax.plot([0, val], [y, y], linestyle='--', color='black',
                 linewidth=1.4, alpha=0.65, zorder=1)
        ax.errorbar(val, y, xerr=std, fmt='none', color='black',
                    capsize=4, capthick=1.2, elinewidth=1.2, zorder=2, alpha=0.65)
        ax.scatter(val, y, color='black', s=160,
                   zorder=3, edgecolors='white', linewidths=1.0)

    display_keys = [label_map.get(k, k) for k in keys] if label_map else keys
    ax.set_yticks(ypos)
    ax.set_yticklabels(display_keys, fontsize=18, fontfamily=font_family)
    ax.set_xlabel(xlabel, fontsize=20, fontfamily=font_family)
    ax.set_xlim(left=0, right=(vals + stds).max() * 1.22)
    ax.set_ylim(-0.6, n - 0.4)
    spine_cleanup(ax)
    ax.xaxis.grid(True, linestyle=':', linewidth=0.5, color='#cccccc', zorder=0)
    ax.set_axisbelow(True)
    ax.tick_params(axis='x', labelsize=18)

# =============================================================================
# FIT FINAL MODELS + COMPUTE %IncMSE PERMUTATION IMPORTANCE
# =============================================================================

print(f"\n{'='*70}\nFINAL MODELS (Optuna, {OPTUNA_TRIALS_FINAL} trials)\n{'='*70}")

final_models = {}
panel_data   = []
t0 = time.time()

for r in RESPONSES:
    X_full = df_model[COVARIATE_NAMES].values
    y_full = df_model[r].values.astype(np.float64)

    best_params = tune_rf(X_full, y_full, OPTUNA_TRIALS_FINAL)

    # Take n_estimators out of best_params so it isn't passed twice
    n_estimators_final = max(best_params.pop('n_estimators', 500), 500)

    model = RandomForestRegressor(**best_params, n_estimators=n_estimators_final)
    model.fit(X_full, y_full)
    final_models[r] = model

    import joblib
    joblib_path = os.path.join(RESULTS_DIR, f'rf_model_{r}.pkl')
    joblib.dump({'model': model, 'features': COVARIATE_NAMES}, joblib_path)
    print(f"  Saved {joblib_path}")

    # -- %IncMSE via permutation importance (analogous to randomForest's %IncMSE) --
    baseline_pred = model.predict(X_full)
    baseline_mse  = mean_squared_error(y_full, baseline_pred)

    print(f"  Computing permutation importance ({N_REPEATS} repeats) for {r} ...")
    perm = permutation_importance(
        model, X_full, y_full,
        n_repeats=N_REPEATS, random_state=SEED,
        scoring='neg_mean_squared_error', n_jobs=N_JOBS_CPU,
    )
    # perm.importances_mean = (base_score - permuted_score) = increase in MSE
    pct_inc_mse     = perm.importances_mean / baseline_mse * 100
    pct_inc_mse_std = perm.importances_std  / baseline_mse * 100

    imp_df = pd.DataFrame({
        'variable': COVARIATE_NAMES,
        'PctIncMSE': pct_inc_mse,
        'PctIncMSE_std': pct_inc_mse_std,
    }).sort_values('PctIncMSE', ascending=False).reset_index(drop=True)
    imp_df.to_csv(os.path.join(RESULTS_DIR, f'varimp_PctIncMSE_{r}.csv'), index=False)

    panel_data.append({
        'tag': r,
        'pct_inc_mse': pct_inc_mse,
        'pct_inc_mse_std': pct_inc_mse_std,
    })
    print(f"  Top 3 ({r}): {imp_df['variable'].iloc[:3].tolist()}")

print(f"Final model fitting + importance finished in {(time.time()-t0)/60:.2f} min")

# =============================================================================
# FIGURE — %IncMSE LOLLIPOP, 2 ROWS x 3 COLUMNS (5 SFI depths, 1 panel hidden)
# =============================================================================

print("\nBuilding %IncMSE lollipop figure ...")

n_resp   = len(RESPONSES)
n_cols   = 3
n_rows   = 2

fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(19, 11),
    gridspec_kw=dict(wspace=0.65, hspace=0.35)
)
axes_flat = axes.flatten()

for idx, pdat in enumerate(panel_data):
    ax  = axes_flat[idx]
    tag = pdat['tag']

    draw_lollipop(ax, COVARIATE_NAMES, pdat['pct_inc_mse'], pdat['pct_inc_mse_std'],
                  TOP_N, xlabel='%IncMSE', label_map=COVARIATE_DISPLAY)

    ax.set_title(RESP_DISPLAY.get(tag, tag),
                 fontweight='bold', fontsize=21,
                 loc='left', pad=10, fontfamily=font_family)
    ax.text(-0.55, 1.07, PANEL_LABELS[idx],
            transform=ax.transAxes,
            fontsize=30, fontweight='bold',
            va='bottom', ha='left',
            fontfamily=font_family, color='#1a1a2e')

for j in range(n_resp, len(axes_flat)):
    axes_flat[j].axis('off')

# =============================================================================
# SAVE
# =============================================================================

out_png = os.path.join(RESULTS_DIR, 'rf_varimp_PctIncMSE_lollipop.png')
out_pdf = os.path.join(RESULTS_DIR, 'rf_varimp_PctIncMSE_lollipop.pdf')

plt.savefig(out_png, dpi=600, bbox_inches='tight',
            facecolor='white', edgecolor='none', pad_inches=0.05)
plt.savefig(out_pdf, format='pdf', bbox_inches='tight',
            facecolor='white', edgecolor='none', pad_inches=0.05)
print(f"Saved: {out_png}")
print(f"Saved: {out_pdf}")
plt.show()

# RK

In [ ]:
"""
RF - Kriging (Regression Kriging) pipeline for the 5 SFI depth-layer
responses (ISF_a .. ISF_e).
    !pip install pykrige -q
"""
!pip install pykrige -q
import os
import json
import time
import joblib
import warnings
import numpy as np
import pandas as pd
import optuna
from pyproj import Transformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from pykrige.ok import OrdinaryKriging

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)


SOURCE_EPSG_WGS84 = 4326
TARGET_EPSG_UTM   = 32738  
                            

VARIOGRAM_MODELS = ['linear', 'power', 'gaussian', 'spherical', 'exponential', 'hole-effect']

VARIOGRAM_OPTUNA_TRIALS = 200

RK_RESULTS_DIR = os.path.join(RESULTS_DIR, 'RF_Kriging')
os.makedirs(RK_RESULTS_DIR, exist_ok=True)


def calc_metrics(obs, pred):
    obs, pred = np.asarray(obs, dtype=np.float64), np.asarray(pred, dtype=np.float64)
    ok = np.isfinite(obs) & np.isfinite(pred)
    obs, pred = obs[ok], pred[ok]
    if len(obs) < 2:
        return dict(N=len(obs), R2=np.nan, RMSE=np.nan, MAE=np.nan,
                    Bias=np.nan, RPIQ=np.nan, CCC=np.nan)

    rmse = np.sqrt(mean_squared_error(obs, pred))
    mae  = mean_absolute_error(obs, pred)
    r2   = 1 - np.sum((obs - pred) ** 2) / np.sum((obs - obs.mean()) ** 2)
    iqr  = np.percentile(obs, 75) - np.percentile(obs, 25)
    rpiq = iqr / rmse if rmse > 0 else 0.0
    bias = float(np.mean(pred - obs))

    mx, my = obs.mean(), pred.mean()
    vx, vy = obs.var(ddof=0), pred.var(ddof=0)
    sxy    = np.cov(obs, pred, ddof=0)[0, 1]
    ccc_den = vx + vy + (mx - my) ** 2
    ccc = (2 * sxy) / ccc_den if ccc_den > 0 else 0.0

    return dict(N=int(len(obs)), R2=r2, RMSE=rmse, MAE=mae, Bias=bias, RPIQ=rpiq, CCC=ccc)


def get_utm_coords(gdf):
    lon = gdf.geometry.x.values
    lat = gdf.geometry.y.values
    transformer = Transformer.from_crs(
        f"EPSG:{SOURCE_EPSG_WGS84}", f"EPSG:{TARGET_EPSG_UTM}", always_xy=True
    )
    x, y = transformer.transform(lon, lat)
    print(f"Reprojected EPSG:{SOURCE_EPSG_WGS84} (WGS84 lon/lat) -> "
          f"EPSG:{TARGET_EPSG_UTM} (UTM meters) for {len(x)} points")
    return np.column_stack([x, y])

def impute_train_test(X_tr, X_te, covariate_names):
    """Median-impute covariates using TRAIN medians only, applied to both
    train and test. Falls back to 0 for a covariate that is entirely NaN
    in the training split (median itself would be NaN)."""
    tr_df = pd.DataFrame(X_tr, columns=covariate_names).copy()
    te_df = pd.DataFrame(X_te, columns=covariate_names).copy()

    medians = tr_df.median()
    still_na_cols = medians[medians.isna()].index.tolist()
    if still_na_cols:
        medians[still_na_cols] = 0.0

    tr_df = tr_df.fillna(medians)
    te_df = te_df.fillna(medians)
    return tr_df.values, te_df.values
# --------------------------------------------------------------------------- #
# RF HYPERPARAMETER TUNING (Optuna, inner KFold)
# --------------------------------------------------------------------------- #

def make_rf_objective(X_np, y_np, n_inner):
    def objective(trial):
        params = dict(
            n_estimators      = trial.suggest_int('n_estimators', 100, 600),
            max_depth         = trial.suggest_int('max_depth', 5, 30),
            min_samples_split = trial.suggest_int('min_samples_split', 2, 15),
            min_samples_leaf  = trial.suggest_int('min_samples_leaf', 1, 8),
            max_features      = trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
            random_state      = 42,
            n_jobs            = -1,
        )
        inner_kf = KFold(n_splits=n_inner, shuffle=True, random_state=0)
        scores = []
        for tr, va in inner_kf.split(X_np):
            Xtr, Xva = X_np[tr], X_np[va]
            scaler = StandardScaler()
            Xtr = scaler.fit_transform(Xtr)
            Xva = scaler.transform(Xva)
            model = RandomForestRegressor(**params)
            model.fit(Xtr, y_np[tr])
            scores.append(np.sqrt(mean_squared_error(y_np[va], model.predict(Xva))))
        return float(np.mean(scores))
    return objective


def tune_rf(X_np, y_np, n_inner, n_trials):
    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(make_rf_objective(X_np, y_np, n_inner), n_trials=n_trials,
                    n_jobs=N_JOBS_CPU, show_progress_bar=False)
    return {**study.best_params, 'random_state': 42, 'n_jobs': -1}


def get_oof_predictions(X_np, y_np, rf_params, n_inner):
    """Honest in-fold OOF predictions on a TRAIN split, used to build
    residuals for kriging without letting the RF see its own targets."""
    kf = KFold(n_splits=n_inner, shuffle=True, random_state=42)
    oof_pred = np.full(len(y_np), np.nan)
    for tr, va in kf.split(X_np):
        scaler = StandardScaler()
        Xtr = scaler.fit_transform(X_np[tr])
        Xva = scaler.transform(X_np[va])
        model = RandomForestRegressor(**rf_params)
        model.fit(Xtr, y_np[tr])
        oof_pred[va] = model.predict(Xva)
    return oof_pred


# --------------------------------------------------------------------------- #
# VARIOGRAM TUNING (Ordinary Kriging on RF residuals)
# --------------------------------------------------------------------------- #

def build_kriging_model(coords, residuals, params):
    return OrdinaryKriging(
        coords[:, 0], coords[:, 1], residuals,
        variogram_model=params['variogram_model'],
        nlags=params['nlags'], weight=params['weight'],
        enable_plotting=False, verbose=False,
    )


def _fit_predict_kriging(coords_tr, resid_tr, coords_te, params):
    krig = build_kriging_model(coords_tr, resid_tr, params)
    z, ss = krig.execute('points', coords_te[:, 0], coords_te[:, 1])
    return np.asarray(z), np.asarray(ss)


def make_variogram_objective(coords, residuals, n_inner):
    def objective(trial):
        params = dict(
            variogram_model=trial.suggest_categorical('variogram_model', VARIOGRAM_MODELS),
            nlags=trial.suggest_int('nlags', 6, 20),
            weight=trial.suggest_categorical('weight', [True, False]),
        )
        kf = KFold(n_splits=n_inner, shuffle=True, random_state=42)
        rmses = []
        for tr, va in kf.split(coords):
            try:
                z, _ = _fit_predict_kriging(coords[tr], residuals[tr], coords[va], params)
                if np.any(np.isnan(z)):
                    return float('inf')
                rmses.append(np.sqrt(mean_squared_error(residuals[va], z)))
            except Exception:
                return float('inf')
        return float(np.mean(rmses))
    return objective


def tune_variogram(coords, residuals, n_inner, n_trials):
    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(make_variogram_objective(coords, residuals, n_inner),
                    n_trials=n_trials, show_progress_bar=False)
    return {**study.best_params, 'kriging_method': 'ordinary'}, study.best_value


# --------------------------------------------------------------------------- #
# NESTED CV: RF-only vs. RF + KRIGING, per response
# --------------------------------------------------------------------------- #

def run_nested_rk(response, covariates, y_full, X_full, coords_full,
                   outer_k=N_FOLDS_OUTER, inner_k=N_FOLDS_INNER,
                   rf_trials=OPTUNA_TRIALS_CV, variogram_trials=VARIOGRAM_OPTUNA_TRIALS,
                   seed=SEED):
    kf = KFold(n_splits=outer_k, shuffle=True, random_state=seed)
    rf_metrics, rk_metrics = [], []
    oof_rows, best_rf_params_list, best_variogram_list = [], [], []

    for i, (tr_idx, te_idx) in enumerate(kf.split(X_full), 1):
        y_tr, y_te = y_full[tr_idx], y_full[te_idx]
        coords_tr, coords_te = coords_full[tr_idx], coords_full[te_idx]

        # covariate imputation fit on TRAIN ONLY, applied to both splits
        X_tr_raw, X_te_raw = X_full[tr_idx], X_full[te_idx]
        X_tr, X_te = impute_train_test(X_tr_raw, X_te_raw, covariates)

        # 1) tune + fit RF on train, predict test -> RF-only
        rf_params = tune_rf(X_tr, y_tr, inner_k, rf_trials)
        scaler = StandardScaler()
        X_tr_s = scaler.fit_transform(X_tr)
        X_te_s = scaler.transform(X_te)
        rf = RandomForestRegressor(**rf_params)
        rf.fit(X_tr_s, y_tr)
        rf_pred_te = rf.predict(X_te_s)
        m_rf = calc_metrics(y_te, rf_pred_te)
        m_rf['fold'] = i
        rf_metrics.append(m_rf)

        oof_pred_tr = get_oof_predictions(X_tr, y_tr, rf_params, inner_k)
        resid_tr = y_tr - oof_pred_tr
        best_vparams, _ = tune_variogram(coords_tr, resid_tr, inner_k, variogram_trials)
        try:
            z_resid_te, _ = _fit_predict_kriging(coords_tr, resid_tr, coords_te, best_vparams)
        except Exception:
            z_resid_te = np.zeros(len(te_idx))

        rk_pred_te = rf_pred_te + z_resid_te
        m_rk = calc_metrics(y_te, rk_pred_te)
        m_rk['fold'] = i
        rk_metrics.append(m_rk)

        oof_rows.append(pd.DataFrame({
            'fold': i, 'obs': y_te, 'rf_pred': rf_pred_te, 'rk_pred': rk_pred_te
        }))
        best_rf_params_list.append({**rf_params, 'fold': i})
        best_variogram_list.append({**best_vparams, 'fold': i})

        print(f"  {response} - fold {i}/{outer_k} done. "
              f"RF R2={m_rf['R2']:.3f} | RF+RK R2={m_rk['R2']:.3f}")

    return dict(
        rf_metrics=pd.DataFrame(rf_metrics),
        rk_metrics=pd.DataFrame(rk_metrics),
        oof=pd.concat(oof_rows, ignore_index=True),
        rf_params=pd.DataFrame(best_rf_params_list),
        variogram_params=pd.DataFrame(best_variogram_list),
    )


# --------------------------------------------------------------------------- #
# MAIN LOOP OVER THE 5 RESPONSES
# --------------------------------------------------------------------------- #

df_geo = data_final.reset_index(drop=True)
coords_all = get_utm_coords(df_geo)
df_flat = df_geo.drop(columns='geometry')

all_results = {}
summary_rows = []

print(f"\n{'='*70}\nNESTED CV: RF vs RF+KRIGING ({N_FOLDS_OUTER} outer x {N_FOLDS_INNER} inner folds)\n{'='*70}")
t0 = time.time()

for response in RESPONSES:
    keep_mask = df_flat[response].notna()
    n_dropped = (~keep_mask).sum()
    if n_dropped:
        print(f"\n{response}: dropping {n_dropped} rows with missing response (not imputing)")

    sub = df_flat.loc[keep_mask].reset_index(drop=True)
    coords_sub = coords_all[keep_mask.values]

    X_full = sub[COVARIATE_NAMES].values
    y_full = sub[response].values.astype(np.float64)

    out_dir = os.path.join(RK_RESULTS_DIR, f'results_{response}')
    os.makedirs(out_dir, exist_ok=True)

    result = run_nested_rk(response, COVARIATE_NAMES, y_full, X_full, coords_sub)
    all_results[response] = result

    result['oof'].to_csv(os.path.join(out_dir, f'oof_predictions_{response}.csv'), index=False)
    result['rf_metrics'].to_csv(os.path.join(out_dir, f'rf_only_fold_metrics_{response}.csv'), index=False)
    result['rk_metrics'].to_csv(os.path.join(out_dir, f'rf_rk_fold_metrics_{response}.csv'), index=False)
    result['rf_params'].to_csv(os.path.join(out_dir, f'rf_best_params_per_fold_{response}.csv'), index=False)
    result['variogram_params'].to_csv(os.path.join(out_dir, f'variogram_best_params_per_fold_{response}.csv'), index=False)

    metric_keys = ['R2', 'RMSE', 'MAE', 'Bias', 'RPIQ', 'CCC']
    row = {'response': response, 'n_obs': len(y_full)}
    for k in metric_keys:
        row[f'RF_{k}_mean']    = result['rf_metrics'][k].mean()
        row[f'RF_{k}_sd']      = result['rf_metrics'][k].std()
        row[f'RF_RK_{k}_mean'] = result['rk_metrics'][k].mean()
        row[f'RF_RK_{k}_sd']   = result['rk_metrics'][k].std()
    summary_rows.append(row)

print(f"\nNested CV (RF + Kriging) finished in {(time.time() - t0) / 60:.2f} min")

summary_table = pd.DataFrame(summary_rows)
print("\n" + "=" * 70)
print("FINAL SUMMARY -- RF vs RF+KRIGING -- all 5 SFI responses")
print("=" * 70)
print(summary_table.to_string(index=False))
summary_table.to_csv(os.path.join(RK_RESULTS_DIR, 'summary_RF_vs_RK.csv'), index=False)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 13.9 MB/s eta 0:00:0000:010:01
Reprojected EPSG:4326 (WGS84 lon/lat) -> EPSG:32738 (UTM meters) for 307 points

NESTED CV: RF vs RF+KRIGING (5 outer x 3 inner folds)

ISF_a: dropping 1 rows with missing response (not imputing)
  ISF_a - fold 1/5 done. RF R2=0.628 | RF+RK R2=0.647
  ISF_a - fold 2/5 done. RF R2=0.578 | RF+RK R2=0.599
  ISF_a - fold 3/5 done. RF R2=0.698 | RF+RK R2=0.698
  ISF_a - fold 4/5 done. RF R2=0.611 | RF+RK R2=0.595
  ISF_a - fold 5/5 done. RF R2=0.619 | RF+RK R2=0.610

ISF_b: dropping 1 rows with missing response (not imputing)
  ISF_b - fold 1/5 done. RF R2=0.532 | RF+RK R2=0.530
  ISF_b - fold 2/5 done. RF R2=0.608 | RF+RK R2=0.603
  ISF_b - fold 3/5 done. RF R2=0.418 | RF+RK R2=0.492
  ISF_b - fold 4/5 done. RF R2=0.521 | RF+RK R2=0.521
  ISF_b - fold 5/5 done. RF R2=0.528 | RF+RK R2=0.524

ISF_c: dropping 1 rows with missing response (not imputing)
  ISF_c - fold 1/5 done. RF R2=0.374 | RF+RK R2=0.373
  

# Maps prediction RF

In [ ]:
# =============================================================================
# GPU-ACCELERATED SPATIAL PREDICTION (FIL) — SFI DEPTH LAYERS
# =============================================================================

_n_gpus = cp.cuda.runtime.getDeviceCount()
assert GPU_ID < _n_gpus, f"GPU_ID={GPU_ID} but only {_n_gpus} GPU(s) available"
print(f"cuML version: {cuml.__version__}  |  GPUs available: {_n_gpus}  |  Using GPU {GPU_ID}")
cp.cuda.Device(GPU_ID).use()


def build_fil_model(rf_model):
    if RF_N_ESTIMATORS is not None and RF_N_ESTIMATORS < rf_model.n_estimators:
        rf_model.estimators_  = rf_model.estimators_[:RF_N_ESTIMATORS]
        rf_model.n_estimators = RF_N_ESTIMATORS
    with cp.cuda.Device(GPU_ID):
        fil = ForestInference.load_from_sklearn(rf_model)
    return fil


def _fil_predict(fil_model, X_gpu):
    raw = fil_model.predict(X_gpu)
    if hasattr(raw, 'values'):
        raw = raw.values
    if isinstance(raw, np.ndarray):
        raw = cp.asarray(raw)
    return raw.ravel().astype(cp.float32)


def _gpu_predict(fil, X_cpu):
    n_v = X_cpu.shape[0]
    out = np.empty(n_v, dtype=np.float32)
    with cp.cuda.Device(GPU_ID):
        for p0 in range(0, n_v, FIL_BATCH_SIZE):
            p1 = min(p0 + FIL_BATCH_SIZE, n_v)
            X_gpu = cp.asarray(X_cpu[p0:p1])
            out[p0:p1] = cp.asnumpy(_fil_predict(fil, X_gpu))
            del X_gpu
            cp.get_default_memory_pool().free_all_blocks()
    return out


def _align_raster(path, ref_profile):
    with rasterio.open(path) as src:
        if (src.crs == ref_profile['crs'] and src.width == ref_profile['width']
                and src.height == ref_profile['height']
                and src.transform == ref_profile['transform']):
            return path
        dst_path = os.path.join('/kaggle/working/Tanety/aligned',
                                 os.path.basename(path).replace('.tif', '_aligned.tif'))
        os.makedirs(os.path.dirname(dst_path), exist_ok=True)
        with rasterio.open(dst_path, 'w', **{**src.profile, **ref_profile,
                                              'count': src.count, 'dtype': src.profile['dtype']}) as dst:
            for i in range(1, src.count + 1):
                reproject(
                    source=rasterio.band(src, i), destination=rasterio.band(dst, i),
                    src_transform=src.transform, src_crs=src.crs,
                    dst_transform=ref_profile['transform'], dst_crs=ref_profile['crs'],
                    resampling=Resampling.bilinear,
                )
        return dst_path


print(f"\n{'='*70}\nBuilding aligned covariate raster paths\n{'='*70}")
ref_path = os.path.join(CROPPED_DIR, 'chm.tif')
with rasterio.open(ref_path) as ref:
    ref_profile = dict(crs=ref.crs, width=ref.width, height=ref.height, transform=ref.transform)

covariate_paths = {}
for name in COVARIATE_NAMES:
    matches = [f for f in cropped_files if f.lower().startswith(name.lower() + '.')]
    assert matches, f"No raster found for covariate '{name}' in {CROPPED_DIR}"
    covariate_paths[name] = _align_raster(os.path.join(CROPPED_DIR, matches[0]), ref_profile)
print("Aligned raster paths ready:", covariate_paths)

for r in RESPONSES:
    print(f"\n{'='*70}\n  FIL SPATIAL PREDICTION — {r}\n{'='*70}")
    fil = build_fil_model(final_models[r])

    with rasterio.open(ref_path) as ref:
        H, W, transform, crs = ref.height, ref.width, ref.transform, ref.crs
        out_profile = ref.profile.copy()
    out_profile.update(dtype='float32', count=1, nodata=NODATA_OUT,
                        compress='lzw', predictor=2,
                        tiled=True, blockxsize=512, blockysize=512, bigtiff='YES')

    raster_handles = {name: rasterio.open(covariate_paths[name]) for name in COVARIATE_NAMES}
    nodata_by_name = {name: raster_handles[name].nodata for name in COVARIATE_NAMES}

    row_starts = list(range(0, H, TILE_SIZE))
    col_starts = list(range(0, W, TILE_SIZE))
    n_cols_t = len(col_starts)
    tiles = [(ti * n_cols_t + tj + 1, row_starts[ti], col_starts[tj])
             for ti in range(len(row_starts)) for tj in range(len(col_starts))]
    print(f"  Grid {H}x{W}  |  {len(tiles)} tiles ({TILE_SIZE}px)  |  CRS: {crs}")

    out_path = os.path.join(PREDICTION_DIR, f'{r}_prediction_FIL.tif')
    tracking_csv = os.path.join(PREDICTION_DIR, f'tile_tracking_{r}.csv')
    trk_file = open(tracking_csv, 'w', newline='')
    trk_writer = csv.DictWriter(trk_file, fieldnames=[
        'tile_idx', 'row_start', 'col_start', 'n_valid_pixels', 'infer_s', 'total_s', 'status'])
    trk_writer.writeheader()

    t0 = time.time()
    with rasterio.open(out_path, 'w', **out_profile) as h_out:
        pbar = tqdm(total=len(tiles), desc=f'{r} FIL tiles', unit='tile', ncols=95)
        for tidx, rs, cs in tiles:
            t_tile = time.time()
            re = min(rs + TILE_SIZE, H); ce = min(cs + TILE_SIZE, W)
            th = re - rs; tw = ce - cs
            win = Window(cs, rs, tw, th)
            tile_arr = np.full((th, tw), NODATA_OUT, np.float32)
            n_v, infer_s, status = 0, 0.0, 'nodata'
            try:
                bands, valid = {}, np.ones((th, tw), dtype=bool)
                for name in COVARIATE_NAMES:
                    arr = raster_handles[name].read(1, window=win).astype(np.float32)
                    nd = nodata_by_name[name]
                    msk = ~np.isfinite(arr)
                    if nd is not None:
                        msk |= np.isclose(arr, nd)
                    valid &= ~msk
                    bands[name] = arr
                n_v = int(valid.sum())
                if n_v > 0:
                    flat = np.where(valid.ravel())[0].astype(np.int64)
                    X = np.column_stack(
                        [bands[name].ravel()[flat] for name in COVARIATE_NAMES]
                    ).astype(np.float32)
                    t_i = time.time()
                    preds = _gpu_predict(fil, X)
                    infer_s = time.time() - t_i
                    r2, c2 = np.unravel_index(flat, (th, tw))
                    tile_arr[r2, c2] = preds
                    status = 'ok'
            except Exception as e:
                status = f'error:{e}'
            h_out.write(tile_arr, 1, window=win)
            del tile_arr
            gc.collect()
            trk_writer.writerow(dict(
                tile_idx=tidx, row_start=rs, col_start=cs, n_valid_pixels=n_v,
                infer_s=f'{infer_s:.2f}', total_s=f'{time.time()-t_tile:.2f}', status=status))
            trk_file.flush()
            pbar.update(1)
            if status not in ('ok', 'nodata'):
                tqdm.write(f"  ERROR tile {tidx}: {status}")
        pbar.close()
    trk_file.close()

    for h in raster_handles.values():
        h.close()
    with cp.cuda.Device(GPU_ID):
        cp.get_default_memory_pool().free_all_blocks()

    print(f"  Done in {(time.time()-t0)/60:.1f} min -> {out_path}")

print(f"\nAll prediction rasters saved to: {PREDICTION_DIR}")

# Scatterplots

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.font_manager as fm
from scipy.stats import gaussian_kde

# =============================================================================
# CONFIGURATION
# =============================================================================

ARIAL_PATH = '/kaggle/input/datasets/hammaadali/arial-font/arial.ttf'

RESP_DISPLAY = {
    'ISF_a': r'SFI$_{0-10}$',
    'ISF_b': r'SFI$_{10-20}$',
    'ISF_c': r'SFI$_{20-30}$',
    'ISF_d': r'SFI$_{30-60}$',
    'ISF_e': r'SFI$_{60-90}$',
}
RESP_UNIT = ''  

COLORMAP     = 'viridis'
PANEL_LABELS = ['a.', 'b.', 'c.', 'd.', 'e.']

FS_BASE       = 22
FS_TICK       = 20
FS_AXLABEL    = 22
FS_TITLE      = 23
FS_PANEL      = 32
FS_STATS      = 19
FS_CBAR_LABEL = 19
FS_CBAR_TICK  = 18

if os.path.exists(ARIAL_PATH):
    fm.fontManager.addfont(ARIAL_PATH)
    font_prop   = fm.FontProperties(fname=ARIAL_PATH)
    font_family = font_prop.get_name()
    print(f"Arial loaded: {font_family}")
else:
    print(f"Arial not found at {ARIAL_PATH} — falling back to sans-serif")
    font_family = 'sans-serif'

plt.rcParams.update({
    'font.family'      : font_family,
    'font.size'        : FS_BASE,
    'axes.labelsize'   : FS_AXLABEL,
    'axes.titlesize'   : FS_TITLE,
    'xtick.labelsize'  : FS_TICK,
    'ytick.labelsize'  : FS_TICK,
    'axes.linewidth'   : 1.4,
    'xtick.major.width': 1.4,
    'ytick.major.width': 1.4,
    'xtick.major.size' : 6,
    'ytick.major.size' : 6,
    'pdf.fonttype'     : 42,
    'ps.fonttype'      : 42,
    'figure.facecolor' : 'white',
    'axes.facecolor'   : 'white',
})

def point_density(x, y):
    mask = ~(np.isnan(x) | np.isnan(y))
    xy   = np.vstack([x[mask], y[mask]])
    try:
        kde     = gaussian_kde(xy)
        density = kde(xy)
    except Exception:
        density = np.ones(mask.sum())
    density = (density - density.min()) / (density.max() - density.min() + 1e-12)
    return density, mask

panel_data = []
for r in RESPONSES:
    obs_pred = cv_results[r]['predictions']       # out-of-fold obs/pred, all folds
    m        = cv_results[r]['metrics']           # per-fold metrics

    panel_data.append({
        'tag':    r,
        'y_true': obs_pred['obs'].values.astype(np.float64),
        'y_pred': obs_pred['pred'].values.astype(np.float64),
        'r2':     m['R2'].mean(),
        'rmse':   m['RMSE'].mean(),
        'rpiq':   m['RPIQ'].mean(),
        'n':      len(obs_pred),
    })


print("Building 2x3 scatter plot ...")

n_resp = len(RESPONSES)
n_cols = 3
n_rows = 2

fig = plt.figure(figsize=(20, 13), dpi=300)
gs  = gridspec.GridSpec(
    n_rows, n_cols, figure=fig,
    left=0.06, right=0.97,
    top=0.93,  bottom=0.08,
    wspace=0.45, hspace=0.25,
)

for idx, pdat in enumerate(panel_data):
    row, col = divmod(idx, n_cols)
    ax = fig.add_subplot(gs[row, col])

    resp_short = RESP_DISPLAY.get(pdat['tag'], pdat['tag'])
    unit       = RESP_UNIT
    y_true     = pdat['y_true']
    y_pred     = pdat['y_pred']

    # Density scatter
    dens, mask = point_density(y_true, y_pred)
    yt, yp     = y_true[mask], y_pred[mask]
    sort_idx   = dens.argsort()
    yt_s       = yt[sort_idx]
    yp_s       = yp[sort_idx]
    dens_s     = dens[sort_idx]

    sc = ax.scatter(yt_s, yp_s,
                    c=dens_s, cmap=COLORMAP,
                    s=20, alpha=0.70, edgecolors='none', rasterized=True)

    # 1:1 line
    lo = min(yt.min(), yp.min())
    hi = max(yt.max(), yp.max())
    ax.plot([lo, hi], [lo, hi], 'k--', linewidth=1.4, alpha=0.65, zorder=1)

    # Regression line
    z     = np.polyfit(yt, yp, 1)
    x_fit = np.linspace(yt.min(), yt.max(), 300)
    ax.plot(x_fit, np.poly1d(z)(x_fit),
            color='red', linewidth=1.8, alpha=0.85, zorder=2)

    # Axis labels
    xlabel = f'Observed {resp_short}' + (f' ({unit})' if unit else '')
    ylabel = f'Predicted {resp_short}' + (f' ({unit})' if unit else '')
    ax.set_xlabel(xlabel, fontsize=FS_AXLABEL, fontfamily=font_family)
    ax.set_ylabel(ylabel, fontsize=FS_AXLABEL, fontfamily=font_family)

    # Title
    title = resp_short + (f' ({unit})' if unit else '')
    ax.set_title(title,
                 fontweight='bold', fontsize=FS_TITLE,
                 loc='left', pad=10, fontfamily=font_family)

    # Panel letter
    ax.text(-0.17, 1.08, PANEL_LABELS[idx],
            transform=ax.transAxes,
            fontsize=FS_PANEL, fontweight='bold',
            va='bottom', ha='left',
            fontfamily=font_family, color='#1a1a2e')

    # Spines & ticks
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    for spine in ['left', 'bottom']:
        ax.spines[spine].set_color('black')
        ax.spines[spine].set_linewidth(1.4)
    ax.tick_params(axis='both', colors='black', direction='out',
                   length=6, width=1.4, labelsize=FS_TICK)
    ax.grid(False)

    # Colorbar
    cbar = plt.colorbar(sc, ax=ax, pad=0.02, aspect=20, shrink=0.78)
    cbar.set_label('Point density', rotation=270, labelpad=22,
                   fontsize=FS_CBAR_LABEL, fontfamily=font_family)
    cbar.ax.tick_params(labelsize=FS_CBAR_TICK, width=0.8, length=3,
                        colors='black')
    cbar.outline.set_linewidth(0.8)
    cbar.outline.set_edgecolor('black')

    # Stats box
    stats_txt = (
        f"$R^2$    = {pdat['r2']:.3f}\n"
        f"RMSE = {pdat['rmse']:.2f}\n"
        f"RPIQ  = {pdat['rpiq']:.2f}\n"
        f"$n$      = {pdat['n']:,}"
    )
    ax.text(0.05, 0.95, stats_txt,
            transform=ax.transAxes,
            fontsize=FS_STATS, verticalalignment='top',
            fontfamily=font_family, color='black',
            linespacing=1.7)

    ax.set_aspect('equal', adjustable='box')

total_slots = n_rows * n_cols
for j in range(n_resp, total_slots):
    row, col = divmod(j, n_cols)
    fig.add_subplot(gs[row, col]).axis('off')

# =============================================================================
# SAVE
# =============================================================================

out_png = os.path.join(RESULTS_DIR, 'rf_scatter_2x3_SFI.png')
out_pdf = os.path.join(RESULTS_DIR, 'rf_scatter_2x3_SFI.pdf')

plt.savefig(out_png, dpi=600, bbox_inches='tight',
            facecolor='white', edgecolor='none', pad_inches=0.05)
plt.savefig(out_pdf, format='pdf', bbox_inches='tight',
            facecolor='white', edgecolor='none', pad_inches=0.05)
print(f"\nSaved PNG : {out_png}")
print(f"Saved PDF : {out_pdf}")
plt.show()

In [ ]:
"""
ISF PREDICTION MAPS BY REGION
"""

import os
import glob
import numpy as np
import geopandas as gpd
import rasterio
from rasterio.mask import mask as rio_mask
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.font_manager as fm
import matplotlib.transforms as mtransforms
import kagglehub

# ===============================================================================
#  CONFIGURATION
# ===============================================================================

BASE_DIR = '/kaggle/working'
AOI_PATH = '/kaggle/input/datasets/antsasarobidyran/area-of-interest/AOI_dissolved.gpkg'
REGION_COL = 'Region'

PRED_PATHS = {
    'ISF_a': '/kaggle/working/Tanety/predictions_FIL/ISF_a_prediction_FIL.tif',
    'ISF_b': '/kaggle/working/Tanety/predictions_FIL/ISF_b_prediction_FIL.tif',
    'ISF_c': '/kaggle/working/Tanety/predictions_FIL/ISF_c_prediction_FIL.tif',
    'ISF_d': '/kaggle/working/Tanety/predictions_FIL/ISF_d_prediction_FIL.tif',
    'ISF_e': '/kaggle/working/Tanety/predictions_FIL/ISF_e_prediction_FIL.tif',
}

# Split across the two figures
FIG1_ORDER = ['ISF_a', 'ISF_b', 'ISF_c']   # 3 rows x 3 cols
FIG2_ORDER = ['ISF_d', 'ISF_e']            # 2 rows x 3 cols

OUTPUT_DIR = BASE_DIR
NODATA_OUT = -9999.0

DPI       = 600
N_BINS_ISF = 3
P_LOW     = 2.0
P_HIGH    = 98.0
ROUND_TO  = 0.05   
DECIMALS  = 2   

_font_path = kagglehub.dataset_download("hammaadali/arial-font")
_ttf_files = glob.glob(f"{_font_path}/**/*.ttf", recursive=True)
if not _ttf_files:
    raise FileNotFoundError(f"No .ttf found under {_font_path}")
arial_ttf   = _ttf_files[0]
fm.fontManager.addfont(arial_ttf)
font_family = fm.FontProperties(fname=arial_ttf).get_name()
print(f"Font loaded: '{font_family}' from {arial_ttf}")

BASE_FS = 12
plt.rcParams.update({
    'font.family'   : font_family,
    'font.size'     : BASE_FS,
    'axes.labelsize': BASE_FS,
    'axes.titlesize': BASE_FS,
    'axes.linewidth': 0.8,
})


_UNIT = r'$\mathregular{(index)}$'   

DEPTH_LABELS = {
    'ISF_a': r'$\mathbf{SFI_{0\mathsf{-}10}}$',
    'ISF_b': r'$\mathbf{SFI_{10\mathsf{-}20}}$',
    'ISF_c': r'$\mathbf{SFI_{20\mathsf{-}30}}$',
    'ISF_d': r'$\mathbf{SFI_{30\mathsf{-}60}}$',
    'ISF_e': r'$\mathbf{SFI_{60\mathsf{-}90}}$',
}

ISF_COLORS = [
    '#D73027', '#FEE08B', '#1A9850',
]

def _make_cmap(name, hex_list):
    c = mcolors.ListedColormap(hex_list, name=name)
    c.set_bad(color='white')
    return c

def make_clean_norm(data_masked, hex_colors, n_bins,
                     p_low=P_LOW, p_high=P_HIGH, round_to=ROUND_TO,
                     decimals=DECIMALS):
    """
    Quantile-based (equal-count) bin edges.
    """
    assert len(hex_colors) == n_bins
    valid = data_masked.compressed()

    # 1) True quantile edges, evenly spaced in percentile space
    pct_points = np.linspace(p_low, p_high, n_bins + 1)
    raw_bounds = np.percentile(valid, pct_points)

    # 2) Round to a clean step for readable labels
    bounds = np.round(raw_bounds / round_to) * round_to

    # 3) Guard against rounding collisions (ties in the data / narrow bins)
    for i in range(1, len(bounds)):
        if bounds[i] <= bounds[i - 1]:
            bounds[i] = bounds[i - 1] + round_to

    # 4) Extend outer edges to the true data extremes so nothing is masked
    bounds[0]  = min(bounds[0],  np.floor(valid.min() / round_to) * round_to)
    bounds[-1] = max(bounds[-1], np.ceil (valid.max() / round_to) * round_to)

    bounds = np.round(bounds, decimals + 4)

    assert len(bounds) - 1 == n_bins, \
        f"Expected {n_bins} bins, got {len(bounds) - 1}"

    cmap_used = _make_cmap('auto', hex_colors)
    norm      = mcolors.BoundaryNorm(bounds, ncolors=n_bins)

    fmt = f'{{:.{decimals}f}}'
    labels = []
    for i in range(n_bins):
        lo, hi = bounds[i], bounds[i + 1]
        if i == 0:
            labels.append(f'< {fmt.format(bounds[1])}')
        elif i == n_bins - 1:
            labels.append(f'> {fmt.format(bounds[-2])}')
        else:
            labels.append(f'{fmt.format(lo)}\u2013{fmt.format(hi)}')
    return norm, cmap_used, labels

print("Loading AOI ...")
aoi = gpd.read_file(AOI_PATH)
assert REGION_COL in aoi.columns, (
    f"'{REGION_COL}' column not found in AOI. Columns present: {list(aoi.columns)}"
)

# Dissolve so each region is a single (multi)polygon, in a stable order
aoi_regions = aoi.dissolve(by=REGION_COL, as_index=False)
region_names = sorted(aoi_regions[REGION_COL].unique().tolist())
assert len(region_names) == 3, (
    f"Expected exactly 3 regions for a 3-column layout, found {len(region_names)}: "
    f"{region_names}"
)
print(f"Regions found: {region_names}")

def crop_raster_to_region(raster_path, region_geom_gdf):
    """
    Mask+crop a raster to a single region's geometry.
    Returns (masked_array, extent=[left, right, bottom, top]).
    """
    with rasterio.open(raster_path) as src:
        geom_proj = region_geom_gdf.to_crs(src.crs)
        geoms     = [g.__geo_interface__ for g in geom_proj.geometry]

        out_arr, out_transform = rio_mask(src, geoms, crop=True, nodata=NODATA_OUT)
        out_arr = out_arr[0].astype(np.float32)
        nodata  = src.nodata if src.nodata is not None else NODATA_OUT

    mask = ~np.isfinite(out_arr)
    mask |= np.isclose(out_arr, nodata)
    mask |= np.isclose(out_arr, NODATA_OUT)

    h, w   = out_arr.shape
    left   = out_transform.c
    top    = out_transform.f
    right  = left + out_transform.a * w
    bottom = top  + out_transform.e * h

    data_masked = np.ma.array(out_arr, mask=mask)
    return data_masked, [left, right, bottom, top]

def draw_panel(ax, data, extent, norm, cmap, letter, title, legend_title, labels):
    ax.set_facecolor('white')
    ax.imshow(
        data,
        cmap=cmap, norm=norm, extent=extent,
        origin='upper', interpolation='nearest',
    )
    ax.set_axis_off()
    orig_bbox      = ax.get_position(original=True)
    cell_transform = mtransforms.BboxTransformTo(orig_bbox) + ax.figure.transFigure

    ax.text(0.01, 1.15, letter,
            transform=cell_transform,
            fontsize=BASE_FS + 9, fontweight='bold',
            color='#111111', ha='left', va='top')

    ax.text(0.12, 1.15, title,
            transform=cell_transform,
            fontsize=BASE_FS + 9, fontweight='bold',
            color='#111111', ha='left', va='top',
            linespacing=1.5)

    legend_patches = [
        mpatches.Patch(
            facecolor=list(cmap.colors)[i], edgecolor='#555555',
            linewidth=0.5, label=labels[i],
        )
        for i in range(len(labels))
    ]
    leg = ax.legend(
        handles=legend_patches,
        title=legend_title,
        title_fontsize=BASE_FS + 1,
        fontsize=BASE_FS + 1,
        loc='lower right',
        bbox_to_anchor=(1.5, 0.03),
        bbox_transform=cell_transform,
        frameon=False,
        handlelength=1.5, handleheight=1.0,
        borderpad=0.65, labelspacing=0.25,
    )
    leg.get_title().set_fontweight('bold')
    leg.get_title().set_multialignment('left')

# ===============================================================================
#  CROP ALL RASTERS (ONCE) TO ALL REGIONS
# ===============================================================================

ALL_RESPONSES = FIG1_ORDER + FIG2_ORDER

print("\nCropping rasters to each region ...")
data_by_resp_region   = {r: {} for r in ALL_RESPONSES}
extent_by_resp_region = {r: {} for r in ALL_RESPONSES}

for r in ALL_RESPONSES:
    raster_path = PRED_PATHS[r]
    for region in region_names:
        region_gdf = aoi_regions[aoi_regions[REGION_COL] == region]
        data_masked, extent = crop_raster_to_region(raster_path, region_gdf)
        data_by_resp_region[r][region]   = data_masked
        extent_by_resp_region[r][region] = extent
        valid = data_masked.compressed()
        if valid.size:
            print(f"  {r} / {region}: n_valid={valid.size:,}  "
                  f"range=[{valid.min():.2f}, {valid.max():.2f}]")
        else:
            print(f"  {r} / {region}: NO VALID PIXELS")
print("\nBuilding shared quantile-based color scale per response variable ...")
norms, cmaps, labels_by_resp = {}, {}, {}
for r in ALL_RESPONSES:
    combined = np.ma.concatenate(
        [data_by_resp_region[r][region].compressed() for region in region_names]
    )
    combined_masked = np.ma.array(combined, mask=np.zeros_like(combined, dtype=bool))
    n, c, l = make_clean_norm(combined_masked, ISF_COLORS, N_BINS_ISF)
    norms[r], cmaps[r], labels_by_resp[r] = n, c, l
    print(f"  {r} bins: {l}")

def build_figure(response_order, letter_start, out_name, fig_h_per_row=5.5):
    n_rows = len(response_order)
    fig_w  = 18.0
    fig_h  = fig_h_per_row * n_rows

    fig, axes = plt.subplots(
        n_rows, 3,
        figsize=(fig_w, fig_h),
        facecolor='white',
        squeeze=False,
        gridspec_kw={'wspace': 0.45, 'hspace': 0.15},
    )

    letters = [chr(ord('a') + letter_start + i) + '.' for i in range(n_rows * 3)]

    for row, r in enumerate(response_order):
        lbl = DEPTH_LABELS[r]
        row_letters = letters[row * 3:(row + 1) * 3]
        for col, region in enumerate(region_names):
            draw_panel(
                ax           = axes[row, col],
                data         = data_by_resp_region[r][region],
                extent       = extent_by_resp_region[r][region],
                norm         = norms[r],
                cmap         = cmaps[r],
                letter       = row_letters[col],
                title        = f'{lbl}\n{region}',
                legend_title = f'Predicted {lbl}\n{_UNIT}',
                labels       = labels_by_resp[r],
            )

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    for ext in ('png', 'pdf'):
        out = os.path.join(OUTPUT_DIR, f'{out_name}.{ext}')
        fig.savefig(out, dpi=DPI, bbox_inches='tight', facecolor='white')
        print(f"Saved -> {out}")

    plt.show()
    plt.close(fig)
    return letters[-1]


print("\nBuilding Figure 1 (3x3) ...")
build_figure(FIG1_ORDER, letter_start=0, out_name='ISF_by_region_3x3')

print("\nBuilding Figure 2 (2x3) ...")
build_figure(FIG2_ORDER, letter_start=len(FIG1_ORDER) * 3, out_name='ISF_by_region_2x3')

In [ ]:
"""
ISF PREDICTION MAPS BY REGION — single figure
"""

import os
import glob
import numpy as np
import geopandas as gpd
import rasterio
from rasterio.mask import mask as rio_mask
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.font_manager as fm
import matplotlib.transforms as mtransforms
import kagglehub

# ===============================================================================
#  CONFIGURATION
# ===============================================================================

BASE_DIR = '/kaggle/working'

AOI_PATH = '/kaggle/input/datasets/antsasarobidyran/area-of-interest/AOI_dissolved.gpkg'
REGION_COL = 'Region'
PRED_PATHS = {
    'ISF_a': '/kaggle/working/Tanety/predictions_FIL/ISF_a_prediction_FIL.tif',
    'ISF_b': '/kaggle/working/Tanety/predictions_FIL/ISF_b_prediction_FIL.tif',
    'ISF_c': '/kaggle/working/Tanety/predictions_FIL/ISF_c_prediction_FIL.tif',
    'ISF_d': '/kaggle/working/Tanety/predictions_FIL/ISF_d_prediction_FIL.tif',
    'ISF_e': '/kaggle/working/Tanety/predictions_FIL/ISF_e_prediction_FIL.tif',
}

# Single figure: 5 rows x 3 columns
RESPONSE_ORDER = ['ISF_a', 'ISF_b', 'ISF_c', 'ISF_d', 'ISF_e']

OUTPUT_DIR = BASE_DIR
NODATA_OUT = -9999.0

DPI       = 600
N_BINS_ISF = 3
P_LOW     = 2.0
P_HIGH    = 98.0
ROUND_TO  = 0.05   # rounding granularity for clean bin-edge labels
DECIMALS  = 2      # decimal places used when formatting bin-edge labels

_font_path = kagglehub.dataset_download("hammaadali/arial-font")
_ttf_files = glob.glob(f"{_font_path}/**/*.ttf", recursive=True)
if not _ttf_files:
    raise FileNotFoundError(f"No .ttf found under {_font_path}")
arial_ttf   = _ttf_files[0]
fm.fontManager.addfont(arial_ttf)
font_family = fm.FontProperties(fname=arial_ttf).get_name()
print(f"Font loaded: '{font_family}' from {arial_ttf}")

BASE_FS = 12
plt.rcParams.update({
    'font.family'   : font_family,
    'font.size'     : BASE_FS,
    'axes.labelsize': BASE_FS,
    'axes.titlesize': BASE_FS,
    'axes.linewidth': 0.8,
})

_UNIT = r'$\mathregular{(index)}$'  

DEPTH_LABELS = {
    'ISF_a': r'$\mathbf{SFI_{0\mathsf{-}10}}$',
    'ISF_b': r'$\mathbf{SFI_{10\mathsf{-}20}}$',
    'ISF_c': r'$\mathbf{SFI_{20\mathsf{-}30}}$',
    'ISF_d': r'$\mathbf{SFI_{30\mathsf{-}60}}$',
    'ISF_e': r'$\mathbf{SFI_{60\mathsf{-}90}}$',
}

ISF_COLORS = [
    '#D73027', '#FEE08B', '#1A9850',
]

def _make_cmap(name, hex_list):
    c = mcolors.ListedColormap(hex_list, name=name)
    c.set_bad(color='white')
    return c

def make_clean_norm(data_masked, hex_colors, n_bins,
                     p_low=P_LOW, p_high=P_HIGH, round_to=ROUND_TO,
                     decimals=DECIMALS):
    """
    Quantile-based (equal-count) bin edges.
    """
    assert len(hex_colors) == n_bins
    valid = data_masked.compressed()

    # 1) True quantile edges, evenly spaced in percentile space
    pct_points = np.linspace(p_low, p_high, n_bins + 1)
    raw_bounds = np.percentile(valid, pct_points)

    # 2) Round to a clean step for readable labels
    bounds = np.round(raw_bounds / round_to) * round_to

    # 3) Guard against rounding collisions (ties in the data / narrow bins)
    for i in range(1, len(bounds)):
        if bounds[i] <= bounds[i - 1]:
            bounds[i] = bounds[i - 1] + round_to

    # 4) Extend outer edges to the true data extremes so nothing is masked
    bounds[0]  = min(bounds[0],  np.floor(valid.min() / round_to) * round_to)
    bounds[-1] = max(bounds[-1], np.ceil (valid.max() / round_to) * round_to)

    bounds = np.round(bounds, decimals + 4)

    assert len(bounds) - 1 == n_bins, \
        f"Expected {n_bins} bins, got {len(bounds) - 1}"

    cmap_used = _make_cmap('auto', hex_colors)
    norm      = mcolors.BoundaryNorm(bounds, ncolors=n_bins)

    fmt = f'{{:.{decimals}f}}'
    labels = []
    for i in range(n_bins):
        lo, hi = bounds[i], bounds[i + 1]
        if i == 0:
            labels.append(f'< {fmt.format(bounds[1])}')
        elif i == n_bins - 1:
            labels.append(f'> {fmt.format(bounds[-2])}')
        else:
            labels.append(f'{fmt.format(lo)}\u2013{fmt.format(hi)}')
    return norm, cmap_used, labels

# ===============================================================================
#  LOAD AOI, GET REGION LIST
# ===============================================================================

print("Loading AOI ...")
aoi = gpd.read_file(AOI_PATH)
assert REGION_COL in aoi.columns, (
    f"'{REGION_COL}' column not found in AOI. Columns present: {list(aoi.columns)}"
)

# Dissolve so each region is a single (multi)polygon, in a stable order
aoi_regions = aoi.dissolve(by=REGION_COL, as_index=False)
region_names = sorted(aoi_regions[REGION_COL].unique().tolist())
assert len(region_names) == 3, (
    f"Expected exactly 3 regions for a 3-column layout, found {len(region_names)}: "
    f"{region_names}"
)
print(f"Regions found: {region_names}")

# ===============================================================================
#  CROP A RASTER TO ONE REGION GEOMETRY
# ===============================================================================

def crop_raster_to_region(raster_path, region_geom_gdf):
    """
    Mask+crop a raster to a single region's geometry.
    Returns (masked_array, extent=[left, right, bottom, top]).
    """
    with rasterio.open(raster_path) as src:
        geom_proj = region_geom_gdf.to_crs(src.crs)
        geoms     = [g.__geo_interface__ for g in geom_proj.geometry]

        out_arr, out_transform = rio_mask(src, geoms, crop=True, nodata=NODATA_OUT)
        out_arr = out_arr[0].astype(np.float32)
        nodata  = src.nodata if src.nodata is not None else NODATA_OUT

    mask = ~np.isfinite(out_arr)
    mask |= np.isclose(out_arr, nodata)
    mask |= np.isclose(out_arr, NODATA_OUT)

    h, w   = out_arr.shape
    left   = out_transform.c
    top    = out_transform.f
    right  = left + out_transform.a * w
    bottom = top  + out_transform.e * h

    data_masked = np.ma.array(out_arr, mask=mask)
    return data_masked, [left, right, bottom, top]

# ===============================================================================
#  DRAW ONE PANEL (same style as the SOC mapping script)
# ===============================================================================

def draw_panel(ax, data, extent, norm, cmap, letter, title, legend_title, labels):
    ax.set_facecolor('white')
    ax.imshow(
        data,
        cmap=cmap, norm=norm, extent=extent,
        origin='upper', interpolation='nearest',
    )
    ax.set_axis_off()
    orig_bbox      = ax.get_position(original=True)
    cell_transform = mtransforms.BboxTransformTo(orig_bbox) + ax.figure.transFigure

    ax.text(0.01, 1.15, letter,
            transform=cell_transform,
            fontsize=BASE_FS + 9, fontweight='bold',
            color='#111111', ha='left', va='top')

    ax.text(0.12, 1.15, title,
            transform=cell_transform,
            fontsize=BASE_FS + 9, fontweight='bold',
            color='#111111', ha='left', va='top',
            linespacing=1.5)

    legend_patches = [
        mpatches.Patch(
            facecolor=list(cmap.colors)[i], edgecolor='#555555',
            linewidth=0.5, label=labels[i],
        )
        for i in range(len(labels))
    ]
    leg = ax.legend(
        handles=legend_patches,
        title=legend_title,
        title_fontsize=BASE_FS + 1,
        fontsize=BASE_FS + 1,
        loc='lower right',
        bbox_to_anchor=(1.5, 0.03),
        bbox_transform=cell_transform,
        frameon=False,
        handlelength=1.5, handleheight=1.0,
        borderpad=0.65, labelspacing=0.25,
    )
    leg.get_title().set_fontweight('bold')
    leg.get_title().set_multialignment('left')

# ===============================================================================
#  CROP ALL RASTERS (ONCE) TO ALL REGIONS
# ===============================================================================

print("\nCropping rasters to each region ...")
data_by_resp_region   = {r: {} for r in RESPONSE_ORDER}
extent_by_resp_region = {r: {} for r in RESPONSE_ORDER}

for r in RESPONSE_ORDER:
    raster_path = PRED_PATHS[r]
    for region in region_names:
        region_gdf = aoi_regions[aoi_regions[REGION_COL] == region]
        data_masked, extent = crop_raster_to_region(raster_path, region_gdf)
        data_by_resp_region[r][region]   = data_masked
        extent_by_resp_region[r][region] = extent
        valid = data_masked.compressed()
        if valid.size:
            print(f"  {r} / {region}: n_valid={valid.size:,}  "
                  f"range=[{valid.min():.2f}, {valid.max():.2f}]")
        else:
            print(f"  {r} / {region}: NO VALID PIXELS")

# ── One shared norm per response variable, built from ALL regions combined ──
print("\nBuilding shared quantile-based color scale per response variable ...")
norms, cmaps, labels_by_resp = {}, {}, {}
for r in RESPONSE_ORDER:
    combined = np.ma.concatenate(
        [data_by_resp_region[r][region].compressed() for region in region_names]
    )
    combined_masked = np.ma.array(combined, mask=np.zeros_like(combined, dtype=bool))
    n, c, l = make_clean_norm(combined_masked, ISF_COLORS, N_BINS_ISF)
    norms[r], cmaps[r], labels_by_resp[r] = n, c, l
    print(f"  {r} bins: {l}")

# ===============================================================================
#  BUILD + SAVE THE 5x3 FIGURE
# ===============================================================================

def build_figure(response_order, out_name, fig_h_per_row=5.5):
    n_rows = len(response_order)
    fig_w  = 18.0
    fig_h  = fig_h_per_row * n_rows

    fig, axes = plt.subplots(
        n_rows, 3,
        figsize=(fig_w, fig_h),
        facecolor='white',
        squeeze=False,
        gridspec_kw={'wspace': 0.45, 'hspace': 0.15},
    )

    letters = [chr(ord('a') + i) + '.' for i in range(n_rows * 3)]

    for row, r in enumerate(response_order):
        lbl = DEPTH_LABELS[r]
        row_letters = letters[row * 3:(row + 1) * 3]
        for col, region in enumerate(region_names):
            draw_panel(
                ax           = axes[row, col],
                data         = data_by_resp_region[r][region],
                extent       = extent_by_resp_region[r][region],
                norm         = norms[r],
                cmap         = cmaps[r],
                letter       = row_letters[col],
                title        = f'{lbl}\n{region}',
                legend_title = f'Predicted {lbl}\n{_UNIT}',
                labels       = labels_by_resp[r],
            )

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    for ext in ('png', 'pdf'):
        out = os.path.join(OUTPUT_DIR, f'{out_name}.{ext}')
        fig.savefig(out, dpi=DPI, bbox_inches='tight', facecolor='white')
        print(f"Saved -> {out}")

    plt.show()
    plt.close(fig)

print("\nBuilding Figure (5x3) ...")
build_figure(RESPONSE_ORDER, out_name='ISF_by_region_5x3')

# RFE

In [ ]:
from sklearn.feature_selection import RFECV

RFE_STEP = 1
RFE_MIN_FEATURES = 3


def run_nested_cv_rfe(df, response, covariates, outer_k=N_FOLDS_OUTER,
                       inner_k=N_FOLDS_INNER, n_trials=OPTUNA_TRIALS_CV, seed=SEED,
                       min_features=RFE_MIN_FEATURES, step=RFE_STEP):
    kf = KFold(n_splits=outer_k, shuffle=True, random_state=seed)
    outer_metrics, fold_predictions, best_params_list, selected_features_list = [], [], [], []

    X_all = df[covariates].values
    y_all = df[response].values.astype(np.float64)

    for i, (tr_idx, te_idx) in enumerate(kf.split(X_all), 1):
        X_tr, X_te = X_all[tr_idx], X_all[te_idx]
        y_tr, y_te = y_all[tr_idx], y_all[te_idx]

        inner_kf = KFold(n_splits=inner_k, shuffle=True, random_state=0)
        rfe_estimator = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
        rfecv = RFECV(estimator=rfe_estimator, step=step, cv=inner_kf,
                      scoring='neg_root_mean_squared_error',
                      min_features_to_select=min_features, n_jobs=N_JOBS_CPU)
        rfecv.fit(X_tr, y_tr)

        support = rfecv.support_
        selected = [c for c, s in zip(covariates, support) if s]
        X_tr_sel, X_te_sel = X_tr[:, support], X_te[:, support]

        best_params = tune_rf(X_tr_sel, y_tr, n_trials)
        rf = RandomForestRegressor(**best_params)
        rf.fit(X_tr_sel, y_tr)
        preds = rf.predict(X_te_sel)

        m = calc_metrics(y_te, preds)
        m['fold'] = i
        m['n_features'] = len(selected)
        outer_metrics.append(m)
        fold_predictions.append(pd.DataFrame({'fold': i, 'obs': y_te, 'pred': preds}))
        best_params_list.append({**best_params, 'fold': i})
        selected_features_list.append({
            'fold': i,
            'response': response,
            'n_features': len(selected),
            'features': ';'.join(selected),
        })

        print(f"  {response} - fold {i}/{outer_k} done. R2 = {m['R2']:.3f}, n_features = {len(selected)}")

    return dict(
        metrics=pd.DataFrame(outer_metrics),
        predictions=pd.concat(fold_predictions, ignore_index=True),
        best_params=pd.DataFrame(best_params_list),
        selected_features=pd.DataFrame(selected_features_list),
    )


def feature_selection_frequency(selected_features_df, covariates, outer_k):
    counts = {c: 0 for c in covariates}
    for feats in selected_features_df['features']:
        for f in feats.split(';'):
            if f in counts:
                counts[f] += 1
    freq = pd.DataFrame({
        'feature': list(counts.keys()),
        'times_selected': list(counts.values()),
    })
    freq['selection_rate'] = freq['times_selected'] / outer_k
    return freq.sort_values('selection_rate', ascending=False).reset_index(drop=True)


print(f"\n{'='*70}\nNESTED CV + RFE ({N_FOLDS_OUTER} outer x {N_FOLDS_INNER} inner folds)\n{'='*70}")
t0 = time.time()
cv_rfe_results = {r: run_nested_cv_rfe(df_model, r, COVARIATE_NAMES) for r in RESPONSES}
print(f"Nested CV + RFE finished in {(time.time()-t0)/60:.2f} min")

summary_table_cv_rfe = pd.DataFrame([
    {'response': r,
     **{f'{k}_mean': cv_rfe_results[r]['metrics'][k].mean() for k in ['R2','RMSE','MAE','RPIQ','CCC']},
     **{f'{k}_sd': cv_rfe_results[r]['metrics'][k].std() for k in ['R2','RMSE','MAE','RPIQ','CCC']},
     'n_features_mean': cv_rfe_results[r]['metrics']['n_features'].mean(),
     'n_features_sd': cv_rfe_results[r]['metrics']['n_features'].std()}
    for r in RESPONSES
])
print(summary_table_cv_rfe)
summary_table_cv_rfe.to_csv(os.path.join(RESULTS_DIR, 'cv_rfe_summary.csv'), index=False)

selected_features_table = pd.concat(
    [cv_rfe_results[r]['selected_features'] for r in RESPONSES], ignore_index=True
)
selected_features_table.to_csv(os.path.join(RESULTS_DIR, 'rfe_selected_features_per_fold.csv'), index=False)

feature_freq_table = pd.concat([
    feature_selection_frequency(cv_rfe_results[r]['selected_features'], COVARIATE_NAMES, N_FOLDS_OUTER).assign(response=r)
    for r in RESPONSES
], ignore_index=True)
feature_freq_table.to_csv(os.path.join(RESULTS_DIR, 'rfe_feature_frequency.csv'), index=False)

best_params_table_rfe = pd.concat(
    [cv_rfe_results[r]['best_params'].assign(response=r) for r in RESPONSES], ignore_index=True
).sort_values(['response', 'fold'])
best_params_table_rfe.to_csv(os.path.join(RESULTS_DIR, 'rfe_best_params_per_fold.csv'), index=False)

obs_pred_all_rfe = pd.concat(
    [cv_rfe_results[r]['predictions'].assign(response=r) for r in RESPONSES], ignore_index=True
)
fig, axes = plt.subplots(1, len(RESPONSES), figsize=(9, 4.5))
for ax, r in zip(axes, RESPONSES):
    sub = obs_pred_all_rfe[obs_pred_all_rfe['response'] == r]
    ax.scatter(sub['obs'], sub['pred'], alpha=0.6, color='steelblue')
    lims = [sub[['obs', 'pred']].min().min(), sub[['obs', 'pred']].max().max()]
    ax.plot(lims, lims, linestyle='--', color='red')
    ax.set_title(r); ax.set_xlabel('Observed'); ax.set_ylabel('Predicted')
fig.suptitle('Observed vs Predicted (out-of-fold, nested CV + RFE)')
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# FINAL RF MODELS WITH RFE FEATURE SELECTION + PERMUTATION IMPORTANCE
# =============================================================================

from sklearn.feature_selection import RFECV
from sklearn.model_selection import KFold

RFE_STEP = 1
RFE_MIN_FEATURES = 3
RFE_FINAL_K_FOLDS = N_FOLDS_INNER

print(f"\n{'='*70}\nFINAL MODELS WITH RFE (Optuna, {OPTUNA_TRIALS_FINAL} trials)\n{'='*70}")

final_models = {}
panel_data = []
rfe_selected_final = []
t0 = time.time()

for r in RESPONSES:
    X_full = df_model[COVARIATE_NAMES].values
    y_full = df_model[r].values.astype(np.float64)

    # -- RFE-CV feature selection on the full dataset (final feature set for deployment) --
    print(f"  Running RFECV for {r} ...")
    inner_kf = KFold(n_splits=RFE_FINAL_K_FOLDS, shuffle=True, random_state=0)
    rfe_estimator = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
    rfecv = RFECV(estimator=rfe_estimator, step=RFE_STEP, cv=inner_kf,
                  scoring='neg_root_mean_squared_error',
                  min_features_to_select=RFE_MIN_FEATURES, n_jobs=N_JOBS_CPU)
    rfecv.fit(X_full, y_full)

    support = rfecv.support_
    selected_covariates = [c for c, s in zip(COVARIATE_NAMES, support) if s]
    print(f"  {r}: selected {len(selected_covariates)}/{len(COVARIATE_NAMES)} features -> {selected_covariates}")

    rfe_selected_final.append({
        'response': r,
        'n_features': len(selected_covariates),
        'features': ';'.join(selected_covariates),
    })

    X_sel = df_model[selected_covariates].values

    best_params = tune_rf(X_sel, y_full, OPTUNA_TRIALS_FINAL)

    n_estimators_final = max(best_params.pop('n_estimators', 500), 500)

    model = RandomForestRegressor(**best_params, n_estimators=n_estimators_final)
    model.fit(X_sel, y_full)
    final_models[r] = model

    import joblib
    joblib_path = os.path.join(RESULTS_DIR, f'rf_model_{r}.pkl')
    joblib.dump({'model': model, 'features': selected_covariates}, joblib_path)
    print(f"  Saved {joblib_path}")

    # -- %IncMSE via permutation importance, restricted to selected features --
    baseline_pred = model.predict(X_sel)
    baseline_mse = mean_squared_error(y_full, baseline_pred)

    print(f"  Computing permutation importance ({N_REPEATS} repeats) for {r} ...")
    perm = permutation_importance(
        model, X_sel, y_full,
        n_repeats=N_REPEATS, random_state=SEED,
        scoring='neg_mean_squared_error', n_jobs=N_JOBS_CPU,
    )
    pct_inc_mse = perm.importances_mean / baseline_mse * 100
    pct_inc_mse_std = perm.importances_std / baseline_mse * 100

    imp_df = pd.DataFrame({
        'variable': selected_covariates,
        'PctIncMSE': pct_inc_mse,
        'PctIncMSE_std': pct_inc_mse_std,
    }).sort_values('PctIncMSE', ascending=False).reset_index(drop=True)
    imp_df.to_csv(os.path.join(RESULTS_DIR, f'varimp_PctIncMSE_{r}.csv'), index=False)

    panel_data.append({
        'tag': r,
        'covariates': selected_covariates,
        'pct_inc_mse': pct_inc_mse,
        'pct_inc_mse_std': pct_inc_mse_std,
    })
    print(f"  Top 3 ({r}): {imp_df['variable'].iloc[:3].tolist()}")

rfe_selected_final_table = pd.DataFrame(rfe_selected_final)
rfe_selected_final_table.to_csv(os.path.join(RESULTS_DIR, 'rfe_selected_features_final.csv'), index=False)

print(f"Final model fitting + RFE + importance finished in {(time.time()-t0)/60:.2f} min")

# =============================================================================
# FIGURE — %IncMSE LOLLIPOP, 2 ROWS x 3 COLUMNS (5 SFI depths, 1 panel hidden)
# =============================================================================

print("\nBuilding %IncMSE lollipop figure ...")

n_resp = len(RESPONSES)
n_cols = 3
n_rows = 2

fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(19, 11),
    gridspec_kw=dict(wspace=0.65, hspace=0.35)
)
axes_flat = axes.flatten()

for idx, pdat in enumerate(panel_data):
    ax = axes_flat[idx]
    tag = pdat['tag']

    # each panel now uses only that response's RFE-selected covariates
    draw_lollipop(ax, pdat['covariates'], pdat['pct_inc_mse'], pdat['pct_inc_mse_std'],
                  len(pdat['covariates']), xlabel='%IncMSE', label_map=COVARIATE_DISPLAY)

    ax.set_title(RESP_DISPLAY.get(tag, tag),
                 fontweight='bold', fontsize=21,
                 loc='left', pad=10, fontfamily=font_family)
    ax.text(-0.55, 1.07, PANEL_LABELS[idx],
            transform=ax.transAxes,
            fontsize=30, fontweight='bold',
            va='bottom', ha='left',
            fontfamily=font_family, color='#1a1a2e')

for j in range(n_resp, len(axes_flat)):
    axes_flat[j].axis('off')

# =============================================================================
# SAVE
# =============================================================================

out_png = os.path.join(RESULTS_DIR, 'rf_varimp_PctIncMSE_lollipop_rfe.png')
out_pdf = os.path.join(RESULTS_DIR, 'rf_varimp_PctIncMSE_lollipop_rfe.pdf')

plt.savefig(out_png, dpi=600, bbox_inches='tight',
            facecolor='white', edgecolor='none', pad_inches=0.05)
plt.savefig(out_pdf, format='pdf', bbox_inches='tight',
            facecolor='white', edgecolor='none', pad_inches=0.05)
print(f"Saved: {out_png}")
print(f"Saved: {out_pdf}")
plt.show()